<a href="https://colab.research.google.com/github/KP-365/Fake_news/blob/Kayleb/eval_MCFakeNews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Set up Colab and clone the repository

This clones the project when needed, removes Colab's incompatible `torchao`, and installs the pinned dependencies. Package-install output should finish without an error before you continue.

In [ ]:
import os
from pathlib import Path

if not Path("predict.py").exists():
    !git clone https://github.com/KP-365/Fake_news
    os.chdir("Fake_news")

!python3 -m pip uninstall -y torchao
!python3 -m pip install -q -r requirements.txt
!python3 -m pip uninstall -y torchvision torchaudio
!pip install -q torchvision==0.27.0

fatal: destination path 'Fake_news' already exists and is not an empty directory.
Found existing installation: torchvision 0.27.0
Uninstalling torchvision-0.27.0:
  Successfully uninstalled torchvision-0.27.0


## 2. Load the trained checkpoint

This loads the tokenizer, RoBERTa-LoRA adapter, and classifier head once for the rest of the notebook. The output shows whether inference will run on CPU or GPU.

In [ ]:
from predict import ID_TO_LABEL, MAX_LENGTH, load_model

model, tokenizer, device = load_model()
print(f"Checkpoint loaded on {device}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Checkpoint loaded on cpu


## 3. Rebuild the held-out test split

This repeats the training notebook's cleaning and stratified 70/15/15 split with seed 42, so the test articles do not leak into training. The output shows the test-set size and class balance.

In [ ]:
import os
import re

import kagglehub
import pandas as pd
from sklearn.model_selection import train_test_split

dataset_path = kagglehub.dataset_download(
    "saurabhshahane/fake-news-classification"
)
welfake_df = pd.read_csv(os.path.join(dataset_path, "WELFake_Dataset.csv"))

if "Unnamed: 0" in welfake_df.columns:
    welfake_df = welfake_df.drop(columns=["Unnamed: 0"])

welfake_df["title"] = welfake_df["title"].fillna("")
welfake_df = welfake_df.dropna(subset=["text"])

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    return re.sub(r"\s+", " ", text).strip()

welfake_df["title"] = welfake_df["title"].apply(clean_text)
welfake_df["text"] = welfake_df["text"].apply(clean_text)
welfake_df = welfake_df[welfake_df["text"].str.len() > 0]
welfake_df["content"] = (
    welfake_df["title"] + ". " + welfake_df["text"]
).str.strip(". ")
welfake_df = welfake_df.drop_duplicates(subset=["text"])

train_df, temp_df = train_test_split(
    welfake_df,
    test_size=0.3,
    stratify=welfake_df["label"],
    random_state=42,
)
valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label"],
    random_state=42,
)

print(f"Held-out test examples: {len(test_df):,}")
print(test_df["label"].value_counts().sort_index())

Using Colab cache for faster access to the 'fake-news-classification' dataset.
Held-out test examples: 9,398
label
0    5193
1    4205
Name: count, dtype: int64


## 4. Run Monte Carlo Dropout inference

Standard inference sets every dropout layer to eval mode, so each article gets one fixed prediction. **Monte Carlo Dropout** instead keeps the dropout layers *active* at test time and runs each article through the network `MC_PASSES` times. Every pass drops a different random subset of units, so the model behaves like a small ensemble and returns a *distribution* of predictions per article.

From that distribution we derive:

- **Mean probability** — the average softmax over all passes; its argmax is the MC prediction and its max is the MC confidence.
- **Predictive entropy** — entropy of the mean probability. This is *total* uncertainty (how unsure the ensemble is overall).
- **Expected entropy** — the average of each pass's own entropy. This is the *aleatoric* part (noise the data itself carries).
- **Mutual information** = predictive entropy − expected entropy. This is the *epistemic* part (disagreement between passes — what the model is unsure about and more data could fix).

`enable_mc_dropout` flips only the `Dropout` modules back to train mode while leaving everything else (LayerNorm, the LoRA weights) in eval mode, so the stochasticity comes purely from dropout. Runtime is roughly `MC_PASSES`× a single evaluation — lower `MC_PASSES` for a quick check, raise it for smoother uncertainty estimates.

In [ ]:
import numpy as np
import torch

MC_PASSES = 30
batch_size = 32
eps = 1e-12


def enable_mc_dropout(module):
    """Re-activate only the dropout layers for stochastic forward passes."""
    for submodule in module.modules():
        if isinstance(submodule, torch.nn.Dropout):
            submodule.train()


test_texts = test_df["content"].tolist()
true_labels = test_df["label"].to_numpy()

model.eval()
enable_mc_dropout(model)

mean_probabilities = []
predictive_entropy = []
expected_entropy = []

for start in range(0, len(test_texts), batch_size):
    batch_texts = test_texts[start:start + batch_size]
    encoded = tokenizer(
        batch_texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="pt",
    )
    encoded = {name: tensor.to(device) for name, tensor in encoded.items()}

    passes = []
    with torch.no_grad():
        for _ in range(MC_PASSES):
            logits = model(**encoded).logits
            passes.append(torch.softmax(logits, dim=-1))
    passes = torch.stack(passes)  # [MC_PASSES, batch, classes]

    mean_probs = passes.mean(dim=0)  # [batch, classes]
    total_entropy = -(mean_probs * torch.log(mean_probs + eps)).sum(dim=-1)
    per_pass_entropy = -(passes * torch.log(passes + eps)).sum(dim=-1)
    aleatoric_entropy = per_pass_entropy.mean(dim=0)

    mean_probabilities.append(mean_probs.cpu().numpy())
    predictive_entropy.append(total_entropy.cpu().numpy())
    expected_entropy.append(aleatoric_entropy.cpu().numpy())

mean_probabilities = np.concatenate(mean_probabilities)
predictive_entropy = np.concatenate(predictive_entropy)
expected_entropy = np.concatenate(expected_entropy)
mutual_information = predictive_entropy - expected_entropy

predicted_labels = mean_probabilities.argmax(axis=1)
prediction_confidences = mean_probabilities.max(axis=1)

print(f"Monte Carlo passes per article: {MC_PASSES}")
print(f"Mean predictive entropy: {predictive_entropy.mean():.4f}")
print(f"Mean mutual information (epistemic): {mutual_information.mean():.4f}")

## 5. Evaluate the MC-averaged predictions

This scores the Monte Carlo mean prediction on every held-out article and, for reference, runs one ordinary dropout-off pass. Accuracy is the overall correct share; macro-F1 weights real and fake equally; the confusion matrix uses true labels as rows and predictions as columns. Comparing the two rows shows whether averaging over dropout passes changes the point predictions (it usually shifts accuracy only slightly - the real payoff of MC Dropout is the uncertainty signal, evaluated in the next sections).

In [ ]:
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

# One deterministic (dropout-off) pass for comparison.
model.eval()
deterministic_probabilities = []
for start in range(0, len(test_texts), batch_size):
    batch_texts = test_texts[start:start + batch_size]
    encoded = tokenizer(
        batch_texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="pt",
    )
    encoded = {name: tensor.to(device) for name, tensor in encoded.items()}
    with torch.no_grad():
        probs = torch.softmax(model(**encoded).logits, dim=-1)
    deterministic_probabilities.append(probs.cpu().numpy())

deterministic_probabilities = np.concatenate(deterministic_probabilities)
deterministic_predictions = deterministic_probabilities.argmax(axis=1)

label_ids = sorted(ID_TO_LABEL)
target_names = [ID_TO_LABEL[label_id] for label_id in label_ids]

mc_accuracy = accuracy_score(true_labels, predicted_labels)
mc_macro_f1 = f1_score(true_labels, predicted_labels, average="macro")
det_accuracy = accuracy_score(true_labels, deterministic_predictions)
det_macro_f1 = f1_score(true_labels, deterministic_predictions, average="macro")

print(f"MC Dropout    | Accuracy: {mc_accuracy:.4f}  Macro F1: {mc_macro_f1:.4f}")
print(f"Deterministic | Accuracy: {det_accuracy:.4f}  Macro F1: {det_macro_f1:.4f}")
print("\nPer-class precision and recall (MC Dropout):")
print(classification_report(
    true_labels,
    predicted_labels,
    labels=label_ids,
    target_names=target_names,
    digits=4,
))

matrix = confusion_matrix(true_labels, predicted_labels, labels=label_ids)
print("Confusion matrix (MC Dropout):")
print(matrix)
ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=target_names,
).plot(cmap="Blues")

## 6. Does uncertainty flag the mistakes?

A useful uncertainty estimate should be *higher on the articles the model gets wrong*. This compares the mean predictive entropy of correct vs incorrect predictions, then draws a **rejection (accuracy–coverage) curve**: articles are sorted from most to least certain, and accuracy is recomputed as the most-uncertain cases are progressively handed off to a human instead of being auto-classified. A curve that climbs as coverage drops means the uncertainty signal is worth acting on — e.g. route the top-uncertainty articles for manual review.

In [ ]:
import matplotlib.pyplot as plt

correct = predicted_labels == true_labels
print(
    f"Mean predictive entropy | correct:   {predictive_entropy[correct].mean():.4f}"
)
print(
    f"Mean predictive entropy | incorrect: {predictive_entropy[~correct].mean():.4f}"
)
print(
    f"Mean mutual information | correct:   {mutual_information[correct].mean():.4f}"
)
print(
    f"Mean mutual information | incorrect: {mutual_information[~correct].mean():.4f}"
)

fig, (ax_box, ax_curve) = plt.subplots(1, 2, figsize=(12, 4.5))

ax_box.boxplot(
    [predictive_entropy[correct], predictive_entropy[~correct]],
    labels=["correct", "incorrect"],
    showfliers=False,
)
ax_box.set_ylabel("Predictive entropy")
ax_box.set_title("Uncertainty of correct vs incorrect predictions")

# Rejection curve: most-certain articles first.
order = np.argsort(predictive_entropy)
sorted_correct = correct[order]
retained_accuracy = np.cumsum(sorted_correct) / np.arange(1, len(order) + 1)
coverage = np.arange(1, len(order) + 1) / len(order)

ax_curve.plot(coverage, retained_accuracy, color="steelblue")
ax_curve.axhline(mc_accuracy, color="grey", linestyle="--", label="full-coverage accuracy")
ax_curve.set_xlabel("Coverage (share of articles auto-classified)")
ax_curve.set_ylabel("Accuracy on retained articles")
ax_curve.set_title("Rejection curve (reject most-uncertain first)")
ax_curve.legend()

plt.tight_layout()
plt.show()

## 7. Calibration: do the confidences mean what they say?

A confidence of 0.9 should be right about 90% of the time. This bins predictions by confidence and plots the reliability diagram — bar height is the actual accuracy in each bin, the diagonal is perfect calibration. **Expected Calibration Error (ECE)** is the average gap between confidence and accuracy across bins (lower is better). Averaging over dropout passes usually softens the over-confident spikes seen in a single deterministic pass, so the MC bars should sit closer to the diagonal and its ECE should be the smaller of the two.

In [ ]:
def expected_calibration_error(confidences, correct, n_bins=15):
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (confidences > lo) & (confidences <= hi)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - confidences[mask].mean())
    return ece


def reliability_bins(confidences, correct, n_bins=15):
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    centers, accuracies = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (confidences > lo) & (confidences <= hi)
        centers.append((lo + hi) / 2)
        accuracies.append(correct[mask].mean() if mask.any() else np.nan)
    return np.array(centers), np.array(accuracies)


det_confidences = deterministic_probabilities.max(axis=1)
det_correct = deterministic_predictions == true_labels

mc_ece = expected_calibration_error(prediction_confidences, correct)
det_ece = expected_calibration_error(det_confidences, det_correct)
print(f"ECE | MC Dropout:    {mc_ece:.4f}")
print(f"ECE | Deterministic: {det_ece:.4f}")

mc_centers, mc_acc = reliability_bins(prediction_confidences, correct)
det_centers, det_acc = reliability_bins(det_confidences, det_correct)

width = 1 / len(mc_centers) * 0.4
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], color="grey", linestyle="--", label="perfect calibration")
ax.bar(mc_centers - width / 2, mc_acc, width=width, color="steelblue",
       label=f"MC Dropout (ECE={mc_ece:.3f})")
ax.bar(det_centers + width / 2, det_acc, width=width, color="indianred",
       label=f"Deterministic (ECE={det_ece:.3f})")
ax.set_xlabel("Confidence")
ax.set_ylabel("Accuracy")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Reliability diagram")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## 8. Inspect 10 random articles with their uncertainty

This selects 10 reproducible random test articles with seed 42. Alongside the true and predicted labels, each row shows the MC confidence, predictive entropy (total uncertainty), and mutual information (epistemic uncertainty). High-entropy rows are the ones you would route for manual review; low-entropy rows are safe to auto-classify.

In [ ]:
sample_rng = np.random.default_rng(42)
sample_positions = sample_rng.choice(len(test_df), size=10, replace=False)
sample_predictions = test_df.iloc[sample_positions][["title", "label"]].copy()
sample_predictions["true_label"] = [
    ID_TO_LABEL[label] for label in sample_predictions["label"]
]
sample_predictions["predicted_label"] = [
    ID_TO_LABEL[label] for label in predicted_labels[sample_positions]
]
sample_predictions["confidence"] = prediction_confidences[sample_positions]
sample_predictions["pred_entropy"] = predictive_entropy[sample_positions]
sample_predictions["mutual_info"] = mutual_information[sample_positions]
sample_predictions.drop(columns=["label"]).reset_index(drop=True)